# Энергия и понижение порядка

**Главная мысль занятия.** У уравнения $m\ddot x = -kx$ нет явного времени — и поэтому сохраняется энергия $\dfrac{m\dot x^2}{2} + \dfrac{kx^2}{2} = E$. Это уравнение уже первого порядка: порядок понизился. А на фазовой плоскости $(x, \dot x)$ оно рисует кривую, по которой движется система.

**Как запустить.** Нажмите **Ctrl+F9** (или «Среда выполнения» → «Выполнить всё»). На предупреждение «блокнот создан не Google» ответьте «Всё равно выполнить». Через несколько секунд ниже появятся картинки с кнопками и ползунками.

Всё управление — кнопками и ползунками.

In [ ]:
#@title Вычисления { display-mode: "form" }
# =============================================================
#  Движок: осциллятор, падение в поле тяготения, уравнения Клеро,
#  метод Эйлера. Интерфейс — в следующих ячейках.
# =============================================================
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

C_ONE, C_TWO, C_THREE = '#2a78d6', '#eb6834', '#1baf7a'
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e6e5e0'


def fmt(x, d=2):
    return ('%.*f' % (d, x)).replace('.', ',').replace('-', '−')


def clean(ax):
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    ax.tick_params(colors=INK2)
    ax.grid(color=GRID, lw=0.8)


# ---------- 1. осциллятор m x'' = -k x,  m = 1, k = 4 ----------
M_OSC, K_OSC = 1.0, 4.0
W_OSC = np.sqrt(K_OSC / M_OSC)            # период pi


def oscillator(E, t):
    A = np.sqrt(2 * E / K_OSC)
    x = A * np.sin(W_OSC * t)
    v = A * W_OSC * np.cos(W_OSC * t)
    return x, v


# ---------- 2. тяготение ----------
GM_EARTH, R_EARTH = 3.986e14, 6.371e6        # м^3/с^2, м
V_ESC = np.sqrt(2 * GM_EARTH / R_EARTH)       # м/с
GM_SUN, R_ORBIT = 1.32712e20, 1.496e11        # м^3/с^2, м
TAU_FALL = np.sqrt(R_ORBIT ** 3 / (2 * GM_SUN)) / 86400.0   # сут


def launch_branch(v0, r_max_plot):
    """Скорость v(r) на подъёме: из энергии v^2 = v0^2 - V_ESC^2 (1 - R/r)."""
    r = np.linspace(1.0, r_max_plot, 2000)          # в радиусах Земли
    v2 = (v0 / 1e3) ** 2 - (V_ESC / 1e3) ** 2 * (1 - 1 / r)
    return r, v2


def launch_numbers(v0):
    """Высота подъёма и время подъёма (или скорость вдали)."""
    if v0 >= V_ESC:
        return dict(escape=True, v_inf=np.sqrt(v0 ** 2 - V_ESC ** 2) / 1e3)
    r_max = R_EARTH / (1 - v0 ** 2 / V_ESC ** 2)
    ev = lambda t, z: z[1]
    ev.terminal, ev.direction = True, -1
    s = solve_ivp(lambda t, z: [z[1], -GM_EARTH / z[0] ** 2], [0, 1e8],
                  [R_EARTH, v0], events=ev, rtol=1e-9, atol=1e-3)
    return dict(escape=False, h=(r_max - R_EARTH) / 1e3, t_up=s.t_events[0][0] / 3600)


def fall_day(eta):
    """Падение с покоя: x = r0 cos^2(eta), t = TAU (eta + sin eta cos eta)."""
    return TAU_FALL * (eta + np.sin(eta) * np.cos(eta))


def fall_state(day):
    eta = brentq(lambda e: fall_day(e) - day, 0.0, np.pi / 2)
    x = R_ORBIT * np.cos(eta) ** 2
    v = np.sqrt(2 * GM_SUN * (1 / max(x, 1.0) - 1 / R_ORBIT))
    return x, v


ORBITS = [('Венера', 1.082e11), ('Меркурий', 5.79e10)]


# ---------- 3. уравнения Клеро y = C x + f(C) ----------
# Особое решение — огибающая: x = -f'(C), y = f(C) - C f'(C).
import sympy as _sp
_C = _sp.Symbol('C')


def clairaut(expr):
    """Строка f(C) -> (f, f') как функции numpy, или None."""
    try:
        f = _sp.sympify(expr.replace('^', '**'), locals={'C': _C, 'ln': _sp.log})
        if f.free_symbols - {_C}:
            return None
        raw = (_sp.lambdify(_C, f, 'numpy'), _sp.lambdify(_C, _sp.diff(f, _C), 'numpy'))
        # постоянная производная даёт число, а не массив — приводим к форме C
        wrap = lambda g: (lambda c: np.asarray(g(c), dtype=float) + np.zeros(np.shape(c)))
        return wrap(raw[0]), wrap(raw[1])
    except Exception:
        return None


# ---------- 4. метод Эйлера на осцилляторе x'' = -x ----------
def euler_runs(h, periods):
    n = int(round(periods * 2 * np.pi / h))
    ex, ev = np.empty(n + 1), np.empty(n + 1)
    cx, cv = np.empty(n + 1), np.empty(n + 1)
    ex[0], ev[0], cx[0], cv[0] = 1.0, 0.0, 1.0, 0.0
    for i in range(n):
        ex[i + 1] = ex[i] + h * ev[i]           # явный Эйлер: обе величины по старым
        ev[i + 1] = ev[i] - h * ex[i]
        cv[i + 1] = cv[i] - h * cx[i]           # Эйлер–Кромер: сначала скорость,
        cx[i + 1] = cx[i] + h * cv[i + 1]       # потом координата по новой скорости
    t = h * np.arange(n + 1)
    return t, ex, ev, cx, cv

---

## 1. Энергия переливается

Слева — координата грузика от времени, в середине — та же точка на фазовой плоскости $(x, \dot x)$, справа — кинетическая и потенциальная энергия. Нажмите ▶.

In [ ]:
#@title Виджет: энергия переливается { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

N_FRAMES = 120
T_SHOW = 2 * np.pi / W_OSC * 2            # два периода


class EnergyApp:

    def __init__(self):
        self.w_play = widgets.Play(value=0, min=0, max=N_FRAMES - 1, step=1, interval=90)
        self.w_t = widgets.IntSlider(value=0, min=0, max=N_FRAMES - 1, description='время',
                                     readout=False, layout=widgets.Layout(width='330px'))
        widgets.jslink((self.w_play, 'value'), (self.w_t, 'value'))
        self.w_E = widgets.FloatSlider(value=2.0, min=0.5, max=4.0, step=0.25,
                                       description='энергия E',
                                       style={'description_width': '80px'},
                                       layout=widgets.Layout(width='330px'))
        self.out = widgets.Output()
        self.w_t.observe(lambda ch: self._refresh(), names='value')
        self.w_E.observe(lambda ch: self._refresh(), names='value')
        display(widgets.HBox([self.w_play, self.w_t, self.w_E]), self.out)
        self._refresh()

    def _refresh(self):
        E = self.w_E.value
        t_all = np.linspace(0, T_SHOW, 600)
        x_all, v_all = oscillator(E, t_all)
        t = T_SHOW * self.w_t.value / (N_FRAMES - 1)
        x, v = oscillator(E, np.array([t]))
        x, v = x[0], v[0]
        Ek, Ep = M_OSC * v ** 2 / 2, K_OSC * x ** 2 / 2
        lim_x, lim_v = np.sqrt(2 * 4.0 / K_OSC) * 1.1, np.sqrt(2 * 4.0 / M_OSC) * 1.1
        with self.out:
            clear_output(wait=True)
            fig = plt.figure(figsize=(11.2, 4.0), dpi=88)
            gs = fig.add_gridspec(1, 3, width_ratios=[1.6, 1.0, 0.55], wspace=0.32)
            fig.subplots_adjust(left=0.05, right=0.98, top=0.86, bottom=0.14)
            a1, a2, a3 = (fig.add_subplot(gs[0, i]) for i in range(3))
            a1.plot(t_all, x_all, color=C_ONE, lw=2)
            a1.plot(t, x, 'o', ms=11, color=C_ONE, mec='white', mew=2, zorder=5)
            a1.set_xlim(0, T_SHOW)
            a1.set_ylim(-lim_x, lim_x)
            a1.set_xlabel('t')
            a1.set_title('координата x(t)', loc='left', color=INK, fontsize=11)
            a2.plot(x_all, v_all, color=C_ONE, lw=2)
            a2.plot(x, v, 'o', ms=11, color=C_ONE, mec='white', mew=2, zorder=5)
            a2.set_xlim(-lim_x, lim_x)
            a2.set_ylim(-lim_v, lim_v)
            a2.set_aspect('equal')
            a2.set_xlabel('x')
            a2.set_ylabel('ẋ')
            a2.set_title('фазовая плоскость', loc='left', color=INK, fontsize=11)
            a3.bar([0, 1, 2], [Ek, Ep, Ek + Ep], color=[C_TWO, C_THREE, '#9a9993'],
                   width=0.7)
            a3.set_xticks([0, 1, 2])
            a3.set_xticklabels(['кинет.', 'потенц.', 'сумма'], fontsize=9)
            a3.set_ylim(0, 4.3)
            a3.set_title('энергия', loc='left', color=INK, fontsize=11)
            for a in (a1, a2, a3):
                clean(a)
            a3.grid(False)
            plt.show()


energy_app = EnergyApp()

### Что попробовать

1. **Нажмите ▶ и смотрите на столбики.** Кинетическая и потенциальная энергия перетекают друг в друга, а сумма стоит на месте. Поэтому точка в середине ходит по одной и той же кривой — это и есть уравнение $\dfrac{m\dot x^2}{2} + \dfrac{kx^2}{2} = E$, нарисованное.

2. **Где точка на эллипсе, когда кинетическая энергия максимальна?** А когда она равна нулю — где в этот момент грузик на левом графике?

3. **Увеличьте энергию вдвое.** Эллипс растёт, но форма та же. Во сколько раз вырос размах колебаний?

4. Здесь $m = 1$, $k = 4$. Почему эллипс вытянут по вертикали, а не круг?

---

## 2. Улетит или вернётся

Камень бросили вертикально вверх с поверхности Земли. Тот же приём, что для осциллятора, — домножить на $\dot x$ — даёт энергию, а энергия даёт скорость на каждой высоте. Картинка — фазовая плоскость: расстояние от центра Земли и скорость.

In [ ]:
#@title Виджет: улетит или вернётся { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

R_PLOT = 12.0                               # до 12 радиусов Земли


class LaunchApp:

    def __init__(self):
        self.w_v0 = widgets.FloatSlider(value=8.0, min=2.0, max=15.0, step=0.1,
                                        description='скорость броска, км/с',
                                        style={'description_width': '170px'},
                                        layout=widgets.Layout(width='520px'))
        self.out, self.info = widgets.Output(), widgets.HTML()
        self.w_v0.observe(lambda ch: self._refresh(), names='value')
        display(self.w_v0, self.out, self.info)
        self._refresh()

    def _refresh(self):
        v0 = self.w_v0.value * 1e3
        with self.out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10.4, 4.6), dpi=88)
            fig.subplots_adjust(left=0.07, right=0.98, top=0.9, bottom=0.13)
            for vb in (4, 6, 8, 10, 12, 14):          # фоновое семейство
                r, v2 = launch_branch(vb * 1e3, R_PLOT)
                ok = v2 >= 0
                ax.plot(r[ok], np.sqrt(v2[ok]), color='#d4d3cc', lw=1.2)
                if vb * 1e3 < V_ESC:                   # обратный путь — только у вернувшихся
                    ax.plot(r[ok], -np.sqrt(v2[ok]), color='#d4d3cc', lw=1.2)
            r, v2 = launch_branch(V_ESC, R_PLOT)       # граница
            ax.plot(r, np.sqrt(v2), color=INK2, lw=1.4, ls='--')
            r, v2 = launch_branch(v0, R_PLOT)
            ok = v2 >= 0
            ax.plot(r[ok], np.sqrt(v2[ok]), color=C_ONE, lw=3, label='вверх')
            n = launch_numbers(v0)
            if not n['escape']:
                ax.plot(r[ok], -np.sqrt(v2[ok]), color=C_TWO, lw=3, label='обратно вниз')
                msg = ('Камень <b>вернётся</b>. Поднимется на %s км над поверхностью; '
                       'подъём займёт %s ч, столько же — падение.'
                       % (fmt(n['h'], 0), fmt(n['t_up'], 2 if n['t_up'] < 10 else 1)))
            else:
                msg = ('Камень <b>улетит</b>: скорость нигде не обращается в ноль. '
                       'Вдали от Земли она стремится к %s км/с.' % fmt(n['v_inf'], 2))
            ax.axhline(0, color=INK2, lw=0.8)
            ax.set_xlim(1, R_PLOT)
            ax.set_ylim(-15.5, 15.5)
            ax.set_xlabel('расстояние от центра Земли, в радиусах Земли')
            ax.set_ylabel('скорость, км/с')
            ax.set_title('пунктир — граница между «вернётся» и «улетит»', loc='left',
                         color=INK, fontsize=11)
            ax.legend(loc='upper right', frameon=False)
            clean(ax)
            plt.show()
        self.info.value = "<div style='font-size:14px'>%s</div>" % msg


launch_app = LaunchApp()

### Что попробовать

1. **Двигайте скорость от 2 до 15 км/с.** Пока кривая упирается в ось (скорость ноль) — камень вернётся. Найдите скорость, при которой кривая перестаёт возвращаться. Сравните со своим ответом к Б3(б).

2. **Посмотрите на время подъёма** при 5, 10 и 11 км/с. Почему последние сотни метров в секунду добавляют так много?

3. Эллипс в первом виджете — замкнутая кривая, а здесь кривая для быстрого камня уходит вправо и не замыкается. Что на этой картинке значит «замкнутая кривая» и «незамкнутая»?

In [ ]:
#@title Виджет: Земля остановилась { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

T_END = fall_day(np.pi / 2)


class FallApp:

    def __init__(self):
        self.w_day = widgets.FloatSlider(value=30.0, min=0.0, max=float(np.floor(T_END)),
                                         step=0.5, description='сутки',
                                         style={'description_width': '60px'},
                                         layout=widgets.Layout(width='520px'))
        self.out, self.info = widgets.Output(), widgets.HTML()
        self.w_day.observe(lambda ch: self._refresh(), names='value')
        display(self.w_day, self.out, self.info)
        eta = np.linspace(0, np.pi / 2, 800)
        self.days = fall_day(eta)
        self.dist = R_ORBIT * np.cos(eta) ** 2 / 1e9        # млн км
        self._refresh()

    def _refresh(self):
        d = self.w_day.value
        x, v = fall_state(d)
        with self.out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10.4, 4.4), dpi=88)
            fig.subplots_adjust(left=0.08, right=0.98, top=0.9, bottom=0.13)
            ax.plot(self.days, self.dist, color=C_ONE, lw=2.5)
            for name, a in ORBITS:
                ax.axhline(a / 1e9, color=INK2, lw=0.9, ls=':')
                ax.text(1, a / 1e9 + 2.5, 'орбита: ' + name, color=INK2, fontsize=9)
            ax.plot(d, x / 1e9, 'o', ms=12, color=C_ONE, mec='white', mew=2, zorder=5)
            ax.set_xlim(0, 70)
            ax.set_ylim(0, 160)
            ax.set_xlabel('сутки после остановки')
            ax.set_ylabel('до Солнца, млн км')
            ax.set_title('Земля остановилась на орбите и падает на Солнце', loc='left',
                         color=INK, fontsize=11)
            clean(ax)
            plt.show()
        self.info.value = ("<div style='font-size:14px'>Через %s сут: до Солнца %s млн км, "
                           "скорость %s км/с.</div>"
                           % (fmt(d, 1), fmt(x / 1e9, 1), fmt(v / 1e3, 1)))


fall_app = FallApp()

### Что попробовать

1. **Сначала прогноз:** за какой день Земля пролетит первую половину пути? Потом поставьте ползунок и проверьте.
2. **Почему кривая такая плоская вначале и крутая в конце?** Посмотрите на скорость в первый день и в последний.
3. Пересечения орбит Венеры и Меркурия видно на графике. Когда Земля долетит до Солнца — сравните со своим ответом к Б3(в).

---

## 3. Огибающая рисует себя сама

Уравнение Клеро $y = xy' + f(y')$ имеет общее решение — семейство прямых $y = Cx + f(C)$. Ползунок двигает одну прямую. Включите «все прямые» — и из веера проступит кривая, которой касается каждая прямая: это особое решение.

In [ ]:
#@title Виджет: огибающая { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

PRESETS = {
    '№293 с пары: y = Cx − ln C': dict(f='-ln(C)', c=(0.25, 4.0), x=(0, 4.5), y=(-2.5, 3.5)),
    'y = Cx + √(1 + C²)': dict(f='sqrt(1 + C^2)', c=(-4.0, 4.0), x=(-2.5, 2.5), y=(-1.5, 3.0)),
}


class EnvelopeApp:

    def __init__(self):
        self.w_pre = widgets.ToggleButtons(options=list(PRESETS) + ['своё f(C)'])
        self.w_f = widgets.Text(value='-C^2', description='f(C) =',
                                continuous_update=False, layout=widgets.Layout(width='260px'))
        self.w_c = widgets.FloatSlider(value=0.5, min=0, max=1, step=0.005,
                                       description='прямая', readout=False,
                                       layout=widgets.Layout(width='330px'))
        self.w_all = widgets.Checkbox(value=False, indent=False, description='все прямые')
        self.w_env = widgets.Checkbox(value=False, indent=False,
                                      description='показать особое решение')
        self.out, self.info = widgets.Output(), widgets.HTML()
        for w in (self.w_pre, self.w_f, self.w_c, self.w_all, self.w_env):
            w.observe(lambda ch: self._refresh(), names='value')
        display(self.w_pre, widgets.HBox([self.w_c, self.w_all, self.w_env]), self.w_f,
                self.out, self.info)
        self._refresh()

    def _setup(self):
        if self.w_pre.value in PRESETS:
            p = PRESETS[self.w_pre.value]
            return clairaut(p['f']), p['c'], p['x'], p['y']
        return clairaut(self.w_f.value), (-3.0, 3.0), (-4.0, 4.0), (-4.0, 4.0)

    def _refresh(self):
        fun, (c0, c1), xr, yr = self._setup()
        with self.out:
            clear_output(wait=True)
            if fun is None:
                print('Не получилось разобрать f(C). Пример записи: -C^2, ln(C), sqrt(1 + C^2)')
                return
            f, df = fun
            fig, ax = plt.subplots(figsize=(8.0, 5.4), dpi=88)
            fig.subplots_adjust(left=0.09, right=0.98, top=0.92, bottom=0.1)
            X = np.linspace(*xr, 200)
            with np.errstate(all='ignore'):
                if self.w_all.value:
                    for C in np.linspace(c0, c1, 31):
                        ax.plot(X, C * X + f(C), color='#c9c8c1', lw=1.0)
                C = c0 + (c1 - c0) * self.w_c.value
                ax.plot(X, C * X + f(C), color=C_ONE, lw=2.8, zorder=4)
                xt, yt = -df(C), f(C) - C * df(C)
                if self.w_env.value:
                    cc = np.linspace(c0, c1, 600)
                    ax.plot(-df(cc), f(cc) - cc * df(cc), color=C_TWO, lw=3, zorder=3)
                    if np.isfinite(xt) and np.isfinite(yt):
                        ax.plot(xt, yt, 'o', ms=9, color=C_TWO, mec='white', mew=2, zorder=5)
            ax.set_xlim(*xr)
            ax.set_ylim(*yr)
            ax.set_xlabel('x')
            ax.set_ylabel('y')
            ax.set_title('синяя — одна прямая из общего решения', loc='left', color=INK,
                         fontsize=11)
            clean(ax)
            plt.show()
        self.info.value = ("<div style='font-size:14px'>Прямая с C = %s.</div>" % fmt(C, 2))


envelope_app = EnvelopeApp()

### Что попробовать

1. **Включите «все прямые».** Ни одна прямая не пересекает проступившую кривую — каждая только касается её. Включите «показать особое решение» и подвигайте ползунок: точка касания бежит по кривой.

2. **Почему особое решение не входит в общее?** Оно не прямая — ни при каком $C$ из формулы $y = Cx + f(C)$ его не получить. Но в каждой своей точке оно касается какого-то решения: через эту точку проходят два решения с одним наклоном, единственность ломается.

3. **«своё f(C)»** — проверьте свои ответы к Б5 листа: впишите $f$ и посмотрите, проступает ли огибающая там, где вы её нашли. Есть ли она у уравнения, где все прямые проходят через одну точку?

---

## 4. Эйлер накачивает энергию

Осциллятор $\ddot x = -x$ — тот же, что в первом виджете, но решаем его методом Эйлера с практики 3. Точное решение ходит по окружности. **Явный Эйлер** считает новые $x$ и $\dot x$ по старым; **Эйлер–Кромер** сначала обновляет скорость, а координату — уже по новой скорости. Разница — одна строчка.

In [ ]:
#@title Виджет: Эйлер накачивает энергию { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt


class EulerApp:

    def __init__(self):
        sl = dict(style={'description_width': '120px'}, layout=widgets.Layout(width='360px'))
        self.w_h = widgets.FloatLogSlider(value=0.1, base=10, min=-2, max=np.log10(0.5),
                                          step=0.02, description='шаг h',
                                          readout_format='.3f', **sl)
        self.w_p = widgets.IntSlider(value=3, min=1, max=10, description='периодов', **sl)
        self.w_e = widgets.Checkbox(value=True, indent=False, description='явный Эйлер')
        self.w_c = widgets.Checkbox(value=True, indent=False, description='Эйлер–Кромер')
        self.out, self.info = widgets.Output(), widgets.HTML()
        for w in (self.w_h, self.w_p, self.w_e, self.w_c):
            w.observe(lambda ch: self._refresh(), names='value')
        display(widgets.HBox([self.w_h, self.w_p]), widgets.HBox([self.w_e, self.w_c]),
                self.out, self.info)
        self._refresh()

    def _refresh(self):
        h, P = self.w_h.value, self.w_p.value
        t, ex, ev, cx, cv = euler_runs(h, P)
        Ee, Ec = ex ** 2 + ev ** 2, cx ** 2 + cv ** 2
        with self.out:
            clear_output(wait=True)
            fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.2, 4.8), dpi=88,
                                         gridspec_kw={'width_ratios': [1.0, 1.15]})
            fig.subplots_adjust(left=0.06, right=0.98, top=0.88, bottom=0.12, wspace=0.25)
            th = np.linspace(0, 2 * np.pi, 300)
            a1.plot(np.cos(th), np.sin(th), color=INK2, lw=1.5, ls='--', label='точное')
            if self.w_e.value:
                a1.plot(ex, ev, color=C_TWO, lw=1.6, label='явный Эйлер')
                a2.plot(t, Ee, color=C_TWO, lw=2, label='явный Эйлер')
            if self.w_c.value:
                a1.plot(cx, cv, color=C_ONE, lw=1.6, label='Эйлер–Кромер')
                a2.plot(t, Ec, color=C_ONE, lw=2, label='Эйлер–Кромер')
            a2.axhline(1.0, color=INK2, lw=1.2, ls='--', label='точное')
            lim = max(1.3, 1.1 * np.sqrt(Ee.max())) if self.w_e.value else 1.3
            a1.set_xlim(-lim, lim)
            a1.set_ylim(-lim, lim)
            a1.set_aspect('equal')
            a1.set_xlabel('x')
            a1.set_ylabel('ẋ')
            a1.set_title('фазовая плоскость', loc='left', color=INK, fontsize=11)
            a2.set_xlabel('t')
            a2.set_title('энергия (в начале — 1)', loc='left', color=INK, fontsize=11)
            a2.legend(loc='upper left', frameon=False, fontsize=9)
            for a in (a1, a2):
                clean(a)
            plt.show()
        self.info.value = (
            "<div style='font-size:14px'>Шагов: %d. Энергия в конце: явный Эйлер — "
            "<b>%s</b>, Эйлер–Кромер — <b>%s</b> (точное значение 1).</div>"
            % (len(t) - 1, fmt(Ee[-1], 2 if Ee[-1] < 100 else 0), fmt(Ec[-1], 3)))


euler_app = EulerApp()

### Что попробовать

1. **Шаг 0,1, три периода.** Явный Эйлер раскручивается наружу: каждый шаг энергия умножается на одно и то же число $1 + h^2$. Ошибка за шаг крошечная — но она копится и никогда не гасится.

2. **Десять периодов.** Энергия выросла в сотни раз. А Эйлер–Кромер почти не отходит от окружности при том же шаге.

3. **Уменьшите шаг вдвое.** Во сколько раз уменьшился рост энергии? Помогает — но цена: вдвое больше шагов.

4. **Сравните с практикой 3.** Там Эйлер на спутнике «прощал» ошибки: решения притягивались к одному режиму, и сбитая точка возвращалась. Здесь притяжения нет — на эллипсе все траектории рядом, ни одна не тянет соседнюю, и ошибке некуда деваться.

---

## Вопросы к виджетам

1. **Почему в первом виджете точка ходит по эллипсу, а не по кругу?**

<details><summary>Ответ</summary>

Кривая — это $\dfrac{m\dot x^2}{2} + \dfrac{kx^2}{2} = E$, эллипс с полуосями $\sqrt{2E/k}$ по $x$ и $\sqrt{2E/m}$ по $\dot x$. При $m = 1$, $k = 4$ вторая вдвое длиннее первой. Круг получился бы при $k = m$ — в четвёртом виджете именно так.

</details>

2. **Во сколько раз вырастет размах колебаний, если энергию увеличить вдвое?**

<details><summary>Ответ</summary>

В $\sqrt2$ раз: размах $A = \sqrt{2E/k}$, энергия пропорциональна квадрату размаха.

</details>

3. **Во втором виджете при 11 км/с подъём длится больше 40 часов, а при 10 км/с — меньше трёх. Почему так резко?**

<details><summary>Ответ</summary>

Далеко от Земли притяжение слабое, и камень почти не тормозится — но и скорость у него там почти нулевая. Чем ближе скорость броска к границе, тем дальше камень улетает и тем дольше ползёт с почти нулевой скоростью у вершины. На самой границе вершина уходит на бесконечность, и время подъёма тоже.

</details>

4. **Почему Земля так долго падает вначале и так быстро в конце?**

<details><summary>Ответ</summary>

Вначале скорость нулевая и сила тяготения на орбите Земли слабая — разгон медленный: первую половину пути Земля проходит примерно за 53 суток из 65. Ближе к Солнцу сила растёт как $1/x^2$, и последние десятки миллионов километров пролетаются за дни.

</details>

5. **Почему в четвёртом виджете энергия по явному Эйлеру растёт ровно в $1 + h^2$ раз за шаг?**

<details><summary>Ответ</summary>

Шаг: $x_{n+1} = x_n + h\dot x_n$, $\dot x_{n+1} = \dot x_n - hx_n$. Сложите квадраты: $x_{n+1}^2 + \dot x_{n+1}^2 = (1 + h^2)(x_n^2 + \dot x_n^2)$ — перекрёстные слагаемые сокращаются. За десять периодов при $h = 0{,}1$ это около 630 шагов и рост энергии примерно в 520 раз.

</details>